# Community-state landscapes

Segmentation of every replicate landscape into community states, comparison of state
boundaries between focal-species histories, and the dominant species at inhibitory
streptomycin concentrations.

Outputs: Figure 2, Figures S1-S4, Table S1, and the statistics in
`results/statistics/dominance_at_inhibitory_doses.csv` and
`results/statistics/boundary_change_vs_focal_sensitivity.csv`.

In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

%run 00_setup_and_functions.ipynb

## Segmentation
Every replicate landscape is segmented with the baseline parameters; the mean landscape of
each community type is also segmented under every sensitivity setting (Figure S1).

In [2]:
replicate_runs = {}
for community in COMMUNITY_TYPES:
    for replicate in COMPLETE_REPLICATES[community]:
        grid = composition_grid(community, replicate)
        replicate_runs[(community, replicate)] = {
            "data_array": grid,
            "result": entropy_penalized_spatial_segmentation(grid, BASE_BOUNDARY_PENALTY, BASE_MAX_BC),
        }

mean_runs = {}
for bp, bcmax in SENSITIVITY_SETTINGS:
    for community in COMMUNITY_TYPES:
        mean_runs[(community, bp, bcmax)] = {
            "result": entropy_penalized_spatial_segmentation(composition_grid(community), bp, bcmax)}


def landscapes(richness):
    """State labels of the 16 replicate landscapes of one richness, ancestral history first."""
    anc, evo = {"3": ("ANC_THREE", "EVO_THREE"), "16": ("ANC_SIXTEEN", "EVO_SIXTEEN")}[richness]
    return ([replicate_runs[(anc, r)]["result"]["labels_grid"] for r in COMPLETE_REPLICATES[anc]]
            + [replicate_runs[(evo, r)]["result"]["labels_grid"] for r in COMPLETE_REPLICATES[evo]])

## Number and size of community states
Typical state size is the cell-weighted mean state size, sum(s^2)/N: the size of the state
containing a randomly chosen cell. Holm correction across the three metrics.

In [3]:
STATE_METRICS = {"n_states": "Number of states",
                 "cell_weighted_state_area_cells": "Cell-weighted state area (cells)",
                 "largest_state_fraction": "Largest-state fraction"}
STATE_TEST_SEEDS = {"3": RANDOM_SEED, "16": RANDOM_SEED + 6}

state_df = pd.DataFrame([
    {"community": c, "replicate": r,
     "n_states": obj["result"]["n_clusters"],
     "cell_weighted_state_area_cells": float((np.array(list(obj["result"]["cluster_sizes"].values()), float) ** 2).sum() / EXPECTED_N_CELLS),
     "largest_state_fraction": float(max(obj["result"]["cluster_sizes"].values()) / EXPECTED_N_CELLS)}
    for (c, r), obj in replicate_runs.items()])

state_rows = []
for richness, anc, evo in RICHNESS_PAIRS:
    family = []
    for metric, label in STATE_METRICS.items():
        x = state_df.loc[state_df.community == anc, metric].to_numpy(float)
        y = state_df.loc[state_df.community == evo, metric].to_numpy(float)
        diff, p = exact_mean_difference_test(x, y)
        lo, hi = bootstrap_diff_ci(x, y, seed=STATE_TEST_SEEDS[richness])
        family.append(dict(richness=richness, analysis=label, statistic="evolved - ancestral",
                           ancestral_mean=x.mean(), evolved_mean=y.mean(), estimate=diff,
                           ci95_low=lo, ci95_high=hi, p_exact=p))
    for r, q in zip(family, holm([r["p_exact"] for r in family])):
        r["p_holm"] = q
    state_rows += family

## Boundary configuration
Each landscape is coded as a 0/1 vector over its 108 grid edges (60 between neighbouring
carbon levels, 48 between neighbouring streptomycin levels). Global differences between
histories: exact PERMANOVA on Euclidean distances between these vectors, and on
1 - adjusted Rand index between whole partitions. Region dependence: heterogeneity of the
evolved-minus-ancestral change in boundary incidence over the 64 cells or the 16 carbon
levels, T = sum (d_k - mean d)^2, with the exact allocation null. Regional contrasts and
boundary orientation are Holm-corrected within each family.

In [4]:
CARBON_REGIONS = {"low": slice(0, 5), "intermediate": slice(5, 11), "high": slice(11, 16)}
ALLOC = independent_masks(16, 8).astype(bool)
OBSERVED = np.r_[np.ones(8, bool), np.zeros(8, bool)]

topology = {}
topology_rows = []
for richness, _, _ in RICHNESS_PAIRS:
    labs = landscapes(richness)
    B = np.array([boundary_vector(l) for l in labs])
    D_edges = np.sqrt(((B[:, None, :] - B[None, :, :]) ** 2).sum(2))
    F_e, R2_e, p_e = exact_permanova_distance(D_edges)
    D_part = 1 - np.array([[adjusted_rand(a, b) for b in labs] for a in labs])
    F_p, R2_p, p_p = exact_permanova_distance(D_part)

    jaccard = np.array([[(a * b).sum() / max(((a + b) > 0).sum(), 1) for b in B] for a in B])
    j_within = np.r_[jaccard[:8, :8][np.triu_indices(8, 1)], jaccard[8:, 8:][np.triu_indices(8, 1)]].mean()
    j_between = jaccard[:8, 8:].mean()

    BI = np.array([boundary_incidence(l) for l in labs])        # landscape x carbon x streptomycin

    def heterogeneity(over_carbon):
        def stat(mask):
            d = BI[~mask].mean(0) - BI[mask].mean(0)           # evolved minus ancestral
            if over_carbon:
                d = d.mean(1)
            return ((d - d.mean()) ** 2).sum()
        obs = stat(OBSERVED)
        null = np.array([stat(m) for m in ALLOC])
        return float(obs), float((null >= obs - 1e-12).mean())
    h_cell, h_carbon = heterogeneity(False), heterogeneity(True)

    topology_rows += [
        dict(richness=richness, analysis="Boundary configuration (edge PERMANOVA)", statistic="pseudo-F",
             estimate=F_e, R2=R2_e, p_exact=p_e),
        dict(richness=richness, analysis="Partition (1 - adjusted Rand PERMANOVA)", statistic="pseudo-F",
             estimate=F_p, R2=R2_p, p_exact=p_p),
        dict(richness=richness, analysis=f"Boundary sharing (mean Jaccard): within histories {j_within:.2f}, between histories {j_between:.2f}",
             statistic="between - within", estimate=j_between - j_within),
        dict(richness=richness, analysis="Spatial heterogeneity of boundary change (64 environments)",
             statistic="sum of squared deviations", estimate=h_cell[0], p_exact=h_cell[1]),
        dict(richness=richness, analysis="Heterogeneity of boundary change along carbon (16 levels)",
             statistic="sum of squared deviations", estimate=h_carbon[0], p_exact=h_carbon[1]),
    ]

    family = []
    for k, (region, sl) in enumerate(CARBON_REGIONS.items()):
        x = BI[:8, sl, :].mean((1, 2)); y = BI[8:, sl, :].mean((1, 2))
        diff, p = exact_mean_difference_test(x, y)
        lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + 500 + 10 * int(richness) + k)
        region_label = f"{CARBON_VALUES[sl.start]:.2f}-{CARBON_VALUES[sl.stop - 1]:.2f}"
        family.append(dict(richness=richness, analysis=f"Boundary incidence, {region} carbon ({region_label} g C/L)",
                           statistic="evolved - ancestral", ancestral_mean=x.mean(), evolved_mean=y.mean(),
                           estimate=diff, ci95_low=lo, ci95_high=hi, p_exact=p))
    for r, q in zip(family, holm([r["p_exact"] for r in family])):
        r["p_holm"] = q
    topology_rows += family

    # Boundary orientation: boundary edges per landscape between neighbouring carbon levels
    # and between neighbouring streptomycin levels.
    L = np.array([np.asarray(l) for l in labs])
    E_carbon = (L[:, 1:, :] != L[:, :-1, :]).astype(float)      # 16 x 15 x 4
    E_strep = (L[:, :, 1:] != L[:, :, :-1]).astype(float)       # 16 x 16 x 3
    family = []
    for k, (axis_name, E) in enumerate([("carbon", E_carbon), ("streptomycin", E_strep)]):
        x = E[:8].sum((1, 2)); y = E[8:].sum((1, 2))
        diff, p = exact_mean_difference_test(x, y)
        lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + 700 + 10 * int(richness) + k)
        family.append(dict(richness=richness, analysis=f"Boundaries along the {axis_name} axis (edges per landscape)",
                           statistic="evolved - ancestral", ancestral_mean=x.mean(), evolved_mean=y.mean(),
                           estimate=diff, ci95_low=lo, ci95_high=hi, p_exact=p))
    for r, q in zip(family, holm([r["p_exact"] for r in family])):
        r["p_holm"] = q
    topology_rows += family

    # Change in boundary incidence along carbon with a bootstrap 95% CI (Figure 2C).
    rng = np.random.default_rng(RANDOM_SEED + 900 + int(richness))
    carbon_anc, carbon_evo = BI[:8].mean(2), BI[8:].mean(2)
    boot = np.array([carbon_evo[rng.integers(0, 8, 8)].mean(0) - carbon_anc[rng.integers(0, 8, 8)].mean(0)
                     for _ in range(10000)])
    topology[richness] = dict(
        delta=BI[8:].mean(0) - BI[:8].mean(0),
        carbon_delta=carbon_evo.mean(0) - carbon_anc.mean(0),
        carbon_lo=np.percentile(boot, 2.5, axis=0), carbon_hi=np.percentile(boot, 97.5, axis=0),
        freq_carbon=(E_carbon[:8].mean(0), E_carbon[8:].mean(0)),
        freq_strep=(E_strep[:8].mean(0), E_strep[8:].mean(0)))

## Table S1

In [5]:
columns = ["richness", "analysis", "statistic", "ancestral_mean", "evolved_mean", "estimate", "R2",
           "ci95_low", "ci95_high", "p_exact", "p_holm"]
table_s1 = pd.DataFrame(topology_rows + state_rows).reindex(columns=columns)
table_s1 = table_s1.sort_values("richness", key=lambda r: r.astype(int), kind="stable")
table_s1.to_csv(TABLE_DIR / "Table_S1_community_state_landscape.csv", index=False)
table_s1

,richness,analysis,statistic,ancestral_mean,evolved_mean,estimate,R2,ci95_low,ci95_high,p_exact,p_holm
0,3,Boundary configuration (edge PERMANOVA),pseudo-F,NaN,NaN,6.577941,0.319660,NaN,NaN,0.000155,NaN
1,3,Partition (1 - adjusted Rand PERMANOVA),pseudo-F,NaN,NaN,11.040790,0.440912,NaN,NaN,0.000155,NaN
2,3,Boundary sharing (mean Jaccard): within histor...,between - within,NaN,NaN,-0.227130,NaN,NaN,NaN,NaN,NaN
3,3,Spatial heterogeneity of boundary change (64 e...,sum of squared deviations,NaN,NaN,2.235569,NaN,NaN,NaN,0.000155,NaN
4,3,Heterogeneity of boundary change along carbon ...,sum of squared deviations,NaN,NaN,0.114502,NaN,NaN,NaN,0.007304,NaN
5,3,"Boundary incidence, low carbon (0.07-0.17 g C/L)",evolved - ancestral,0.422396,0.324479,-0.097917,NaN,-0.150000,-0.050521,0.002176,0.006527
6,3,"Boundary incidence, intermediate carbon (0.21-...",evolved - ancestral,0.343750,0.301649,-0.042101,NaN,-0.088108,-0.001736,0.108314,0.216628
7,3,"Boundary incidence, high carbon (0.82-2.00 g C/L)",evolved - ancestral,0.293229,0.270833,-0.022396,NaN,-0.079688,0.035417,0.488112,0.488112
8,3,Boundaries along the carbon axis (edges per la...,evolved - ancestral,15.125000,13.250000,-1.875000,NaN,-4.250000,0.500000,0.210101,0.210101
9,3,Boundaries along the streptomycin axis (edges ...,evolved - ancestral,23.375000,20.750000,-2.625000,NaN,-4.250000,-0.875000,0.021601,0.043201


## Dominant species at inhibitory streptomycin concentrations
Mean relative abundance of *A. caviae* and *C. koseri* at 20 and 200 ug/ml in the
16-species community (carbon-averaged within each landscape); Holm correction across the
four contrasts.

In [6]:
dominance_rows = []
for sp in ["HAMBI_1972", "HAMBI_1287"]:
    for dose in [20.0, 200.0]:
        means = (sample_compositions.loc[sample_compositions.streptomycin == dose]
                 .groupby(["community", "replicate"])[sp].mean())
        x, y = means["ANC_SIXTEEN"].to_numpy(float), means["EVO_SIXTEEN"].to_numpy(float)
        diff, p = exact_mean_difference_test(x, y)
        dominance_rows.append(dict(species=SPECIES_NAMES[sp], streptomycin=dose, ancestral_mean=x.mean(),
                                   evolved_mean=y.mean(), evolved_minus_ancestral=diff, p_exact=p))
dominance = pd.DataFrame(dominance_rows)
dominance["p_holm"] = holm(dominance.p_exact)
dominance.to_csv(STATS_DIR / "dominance_at_inhibitory_doses.csv", index=False)
dominance

,species,streptomycin,ancestral_mean,evolved_mean,evolved_minus_ancestral,p_exact,p_holm
0,Aeromonas caviae HAMBI 1972,20.0,0.001828,0.562445,0.560616,0.000155,0.000622
1,Aeromonas caviae HAMBI 1972,200.0,0.003596,0.667579,0.663983,0.000155,0.000622
2,Citrobacter koseri HAMBI 1287,20.0,0.587037,0.297978,-0.289059,0.000155,0.000622
3,Citrobacter koseri HAMBI 1287,200.0,0.659272,0.273182,-0.386090,0.000155,0.000622


## Boundary change and environmental sensitivity of the focal species
Sensitivity of resistance-evolved *A. caviae* is the magnitude of the local gradient of its
mean relative abundance over the carbon x streptomycin grid (central differences in grid-index
space, one-sided at the edges). It is related to the evolved-minus-ancestral change in
boundary incidence across the 64 cells by Spearman correlation. Both fields are recomputed
for every allocation of history labels (6,435 distinct allocations); the CI resamples
landscapes within history; the partial correlation controls for mean evolved focal abundance.

In [7]:
def local_gradient(P):
    ng, ns = P.shape
    G = np.empty((ng, ns))
    for i in range(ng):
        for j in range(ns):
            i0, i1 = max(i - 1, 0), min(i + 1, ng - 1)
            j0, j1 = max(j - 1, 0), min(j + 1, ns - 1)
            G[i, j] = np.hypot((P[i1, j] - P[i0, j]) / (i1 - i0), (P[i, j1] - P[i, j0]) / (j1 - j0))
    return G


def boundary_sensitivity_association(anc, evo, seed):
    keys = [(anc, r) for r in COMPLETE_REPLICATES[anc]] + [(evo, r) for r in COMPLETE_REPLICATES[evo]]
    bi = [boundary_incidence(replicate_runs[k]["result"]["labels_grid"]) for k in keys]
    fa = [replicate_runs[k]["data_array"][:, :, SPECIES.index(FOCAL_SPECIES)] for k in keys]

    def stat(idx_a, idx_b):
        sensitivity = local_gradient(np.mean([fa[i] for i in idx_b], axis=0))
        change = np.mean([bi[i] for i in idx_b], axis=0) - np.mean([bi[i] for i in idx_a], axis=0)
        return sensitivity.ravel(), change.ravel()

    sens, change = stat(list(range(8)), list(range(8, 16)))
    rho = float(spearmanr(sens, change).statistic)
    allocations = [a for a in combinations(range(16), 8) if 0 in a]
    null = np.array([spearmanr(*stat(list(a), [i for i in range(16) if i not in a])).statistic for a in allocations])
    rng = np.random.default_rng(RANDOM_SEED + seed)
    boot = np.array([spearmanr(*stat(list(rng.choice(8, 8, replace=True)),
                                     list(rng.choice(np.arange(8, 16), 8, replace=True)))).statistic
                     for _ in range(10000)])
    focal_evolved = np.mean(fa[8:], axis=0).ravel()
    return dict(spearman_rho=rho, ci95_low=float(np.percentile(boot, 2.5)), ci95_high=float(np.percentile(boot, 97.5)),
                p_exact=float((np.abs(null) >= abs(rho) - 1e-12).mean()), n_allocations=len(allocations),
                partial_rho_controlling_focal_abundance=partial_spearman(sens, change, focal_evolved))


sensitivity_association = pd.DataFrame([
    dict(richness="16", **boundary_sensitivity_association("ANC_SIXTEEN", "EVO_SIXTEEN", 77)),
    dict(richness="3", **boundary_sensitivity_association("ANC_THREE", "EVO_THREE", 78)),
])
sensitivity_association.to_csv(STATS_DIR / "boundary_change_vs_focal_sensitivity.csv", index=False)
sensitivity_association

,richness,spearman_rho,ci95_low,ci95_high,p_exact,n_allocations,partial_rho_controlling_focal_abundance
0,16,0.208005,0.124397,0.275858,0.245066,6435,0.253614
1,3,0.223923,0.107289,0.319700,0.105361,6435,0.392670


## Figure 2
A, composition of every replicate 16-species landscape with its state boundaries; B, frequency of each boundary across the eight landscapes of each history; C, evolved-minus-ancestral change in boundary incidence along the carbon gradient; D, the same change in every environment.

In [8]:
fig = plt.figure(figsize=(183 * MM, 210 * MM))
outer = fig.add_gridspec(2, 1, height_ratios=[3.35, 1.00], hspace=0.42)

# A: composition of all eight 16-species replicate landscapes of each history (ancestral in
# the two left columns, resistance-evolved in the two right columns) with their state boundaries.
gsa = outer[0].subgridspec(4, 4, wspace=0.14, hspace=0.26)
axes_a = {}
for community, hist_label, first_col in [("ANC_SIXTEEN", "ancestral", 0), ("EVO_SIXTEEN", "resistance-evolved", 2)]:
    for rep in range(1, 9):
        row, col = (rep - 1) % 4, first_col + (rep - 1) // 4
        ax = fig.add_subplot(gsa[row, col])
        axes_a[(community, rep)] = ax
        obj = replicate_runs[(community, rep)]
        plot_composition_grid(ax, obj["data_array"], labels=obj["result"]["labels_grid"],
                              show_x=(row == 3), show_y=(col == 0), ytick_every=2)
        ax.set_xlabel(""); ax.set_ylabel("")
        if col == 0:
            ax.tick_params(axis="y", labelsize=4.0, pad=1.0)
        if row == 3:
            ax.tick_params(axis="x", labelsize=4.5, pad=1.5)
        ax.text(0.5, 1.025, f"Rep. {rep}  |  {hist_label}", transform=ax.transAxes, ha="center", va="bottom", fontsize=4.7)
panel_label(axes_a[("ANC_SIXTEEN", 1)], "a")
fig.text(0.52, 0.985, r"16-species communities containing ancestral or resistance-evolved $\it{Aeromonas\ caviae}$",
         ha="center", va="top", fontsize=7.4)
strep_text = fig.text(0.5, 0.0, STREP_AXIS_LABEL, ha="center", va="center", fontsize=8)
carbon_text = fig.text(0.0, 0.5, CARBON_AXIS_LABEL, rotation=90, ha="center", va="center", fontsize=8)
species_legend = fig.legend(handles=species_legend_handles(SPECIES), loc="upper center", bbox_to_anchor=(0.5, 0.0),
                            ncol=8, frameon=False, fontsize=4.25, columnspacing=0.75, handletextpad=0.28)

gsb = outer[1].subgridspec(1, 3, width_ratios=[1.30, 1.00, 1.30], wspace=0.50)


# B: boundary frequency. Each edge is drawn with width and opacity proportional to the
# fraction of the eight 16-species landscapes in which it separates two states.
def draw_boundary_frequency(ax, f_carbon, f_strep, color):
    segments, weights = [], []
    for i in range(f_carbon.shape[0]):           # edge between carbon levels i and i + 1
        for j in range(f_carbon.shape[1]):
            segments.append([(j - 0.5, i + 0.5), (j + 0.5, i + 0.5)]); weights.append(f_carbon[i, j])
    for i in range(f_strep.shape[0]):            # edge between streptomycin levels j and j + 1
        for j in range(f_strep.shape[1]):
            segments.append([(j + 0.5, i - 0.5), (j + 0.5, i + 0.5)]); weights.append(f_strep[i, j])
    weights = np.asarray(weights); keep = weights > 0
    rgba = np.array([to_rgba(color, alpha=0.25 + 0.75 * w) for w in weights[keep]])
    ax.add_collection(LineCollection([s for s, k in zip(segments, keep) if k], colors=rgba,
                                     linewidths=0.25 + 2.4 * weights[keep], capstyle="butt", zorder=3))
    for yy in np.arange(-0.5, N_CARBON, 1):
        ax.axhline(yy, color="0.90", lw=0.25, zorder=1)
    for xx in np.arange(-0.5, N_STREP, 1):
        ax.axvline(xx, color="0.90", lw=0.25, zorder=1)
    ax.set_xlim(-0.5, N_STREP - 0.5)
    for spine in ax.spines.values():
        spine.set_visible(False)


subb = gsb[0, 0].subgridspec(1, 2, wspace=0.16)
for k, (title, color) in enumerate([("Ancestral", ANCESTRAL_COLOR), ("Resistance-evolved", EVOLVED_COLOR)]):
    ax = fig.add_subplot(subb[0, k])
    if k == 0:
        panel_label(ax, "b", dx=-40)
    draw_boundary_frequency(ax, topology["16"]["freq_carbon"][k], topology["16"]["freq_strep"][k], color)
    format_grid_axis(ax, show_x=True, show_y=(k == 0), xlabel="", ytick_every=3)
    ax.set_title(title, fontsize=6.2, pad=3, color=color)
    ax.tick_params(labelsize=5)
    if k == 0:
        ax.set_ylabel(CARBON_AXIS_LABEL, fontsize=6)
        ax.set_xlabel(STREP_AXIS_LABEL)
        ax.xaxis.set_label_coords(1.08, -0.13)

# C: change in boundary incidence along carbon (mean across streptomycin; bootstrap 95% CI).
axC = fig.add_subplot(gsb[0, 1])
panel_label(axC, "c")
axC.axhline(0, color=REFERENCE_GREY, lw=0.6, zorder=1)
for richness, ls, mk, lab in [("3", "--", "s", "3 species"), ("16", "-", "o", "16 species")]:
    t = topology[richness]
    axC.fill_between(CARBON_VALUES, t["carbon_lo"], t["carbon_hi"], color="#6E6E6E", alpha=0.16, lw=0, zorder=2)
    axC.plot(CARBON_VALUES, t["carbon_delta"], ls=ls, color="#333333", lw=1.0, marker=mk, ms=2.4, zorder=3, label=lab)
axC.set_xscale("log")
axC.set_xlabel(CARBON_AXIS_LABEL)
axC.set_ylabel("Change in boundary incidence\n(evolved minus ancestral)")
axC.legend(fontsize=4.6, frameon=False, loc="lower right", handlelength=2.2)

# D: change in boundary incidence across the landscape.
subd = gsb[0, 2].subgridspec(1, 3, width_ratios=[1, 1, 0.09], wspace=0.22)
lim = max(float(np.max(np.abs(topology[r]["delta"]))) for r in ["3", "16"])
for k, (richness, title) in enumerate([("3", "3 species"), ("16", "16 species")]):
    ax = fig.add_subplot(subd[0, k])
    if k == 0:
        panel_label(ax, "d", dx=-46)
    imD = ax.imshow(topology[richness]["delta"], origin="upper", aspect="auto", cmap=CMAP_HISTORY, vmin=-lim, vmax=lim)
    format_grid_axis(ax, show_x=True, show_y=(k == 0), xlabel="", ytick_every=3)
    ax.set_title(title, fontsize=6.4, pad=3)
    ax.tick_params(labelsize=5)
    if k == 0:
        ax.set_ylabel(CARBON_AXIS_LABEL, fontsize=6)
        ax.set_xlabel(STREP_AXIS_LABEL)
        ax.xaxis.set_label_coords(1.11, -0.13)
cbar = fig.colorbar(imD, cax=fig.add_subplot(subd[0, 2]))
cbar.set_label("Change in boundary incidence", fontsize=5.9, labelpad=2)
cbar.ax.tick_params(labelsize=5.5)

fig.legend(handles=[
    Line2D([0], [0], color=ANCESTRAL_COLOR, lw=1.6, label=r"Ancestral $\it{A.\ caviae}$"),
    Line2D([0], [0], color=EVOLVED_COLOR, lw=1.6, label=r"Resistance-evolved $\it{A.\ caviae}$"),
    Line2D([0], [0], color="0.2", lw=0.25 + 2.4 * 0.25, label="boundary in 2 of 8 landscapes"),
    Line2D([0], [0], color="0.2", lw=0.25 + 2.4 * 1.0, label="boundary in 8 of 8 landscapes")],
    loc="lower left", bbox_to_anchor=(0.075, 0.005), ncol=2, frameon=False, fontsize=5.0,
    handlelength=2.2, columnspacing=1.2)
fig.subplots_adjust(top=0.945, bottom=0.095, left=0.108, right=0.985)

# Shared axis labels and species key positioned from the extent of the panel-A gallery.
positions = [ax.get_position() for ax in axes_a.values()]
a_left, a_right = min(p.x0 for p in positions), max(p.x1 for p in positions)
a_bottom, a_top = min(p.y0 for p in positions), max(p.y1 for p in positions)
strep_text.set_position((0.5 * (a_left + a_right), a_bottom - 0.034))
carbon_text.set_position((a_left - 0.080, 0.5 * (a_bottom + a_top)))
species_legend.set_bbox_to_anchor((0.5 * (a_left + a_right), a_bottom - 0.055), transform=fig.transFigure)
save_figure(fig, "Figure_2_community_state_landscape")

## Figure S1: segmentation sensitivity

In [9]:
fig, axes = plt.subplots(len(SENSITIVITY_SETTINGS), 4, figsize=(183 * MM, 238 * MM), sharex=True, sharey=True, squeeze=False)
for row, (bp, bcmax) in enumerate(SENSITIVITY_SETTINGS):
    for col, c in enumerate(COMMUNITY_TYPES):
        plot_segmentation_map(axes[row, col], mean_runs[(c, bp, bcmax)]["result"], show_x=(row == len(SENSITIVITY_SETTINGS) - 1),
                              show_y=(col == 0), descriptor=(PANEL_TREATMENT_LABELS[c] if row == 0 else None))
        axes[row, col].set_ylabel("")
        axes[row, col].tick_params(axis="y", labelsize=3.7, pad=0.8)
    axes[row, 0].text(-0.58, 0.5, f"Penalty {bp}\nMax BC {bcmax}", transform=axes[row, 0].transAxes, va="center", ha="right", fontsize=5.5)
fig.supylabel(CARBON_AXIS_LABEL, x=0.09, fontsize=7)
dominant = sorted({SPECIES[int(k)] for rr in mean_runs.values() for k in np.unique(dominant_species_map(rr["result"]))}, key=SPECIES.index)
fig.legend(handles=species_legend_handles(dominant), loc="lower center", bbox_to_anchor=(0.52, 0.01), ncol=4, frameon=False,
           fontsize=4.6, columnspacing=0.8)
fig.subplots_adjust(left=0.17, right=0.99, bottom=0.19, top=0.93, hspace=0.18, wspace=0.10)
save_figure(fig, "Figure_S1_segmentation_sensitivity")

## Figure S2: composition of every replicate landscape with its state boundaries

In [10]:
all_reps = sorted(set().union(*[set(v) for v in COMPLETE_REPLICATES.values()]))
fig, axes = plt.subplots(len(all_reps), 4, figsize=(183 * MM, 238 * MM), sharex=True, sharey=True, squeeze=False)
for row, rep in enumerate(all_reps):
    for col, c in enumerate(COMMUNITY_TYPES):
        ax = axes[row, col]
        obj = replicate_runs[(c, rep)]
        plot_composition_grid(ax, obj["data_array"], labels=obj["result"]["labels_grid"], show_x=(row == len(all_reps) - 1),
                              show_y=(col == 0), descriptor=(PANEL_TREATMENT_LABELS[c] if row == 0 else None))
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelsize=3.45, pad=0.7)
        if col == 0:
            ax.text(-0.27, 0.5, f"Rep. {rep}", transform=ax.transAxes, rotation=90, va="center", ha="center", fontsize=5.3)
fig.supylabel(CARBON_AXIS_LABEL, x=0.028, fontsize=7)
fig.legend(handles=species_legend_handles(SPECIES), loc="lower center", bbox_to_anchor=(0.52, 0.01), ncol=4, frameon=False,
           fontsize=4.45, columnspacing=0.75, handletextpad=0.3)
fig.subplots_adjust(left=0.115, right=0.995, bottom=0.20, top=0.93, hspace=0.12, wspace=0.08)
save_figure(fig, "Figure_S2_replicate_composition_with_boundaries")

## Figure S3: replicate segmentations coloured by dominant species

In [11]:
fig, axes = plt.subplots(len(all_reps), 4, figsize=(183 * MM, 238 * MM), sharex=True, sharey=True, squeeze=False)
for row, rep in enumerate(all_reps):
    for col, c in enumerate(COMMUNITY_TYPES):
        ax = axes[row, col]
        plot_segmentation_map(ax, replicate_runs[(c, rep)]["result"], show_x=(row == len(all_reps) - 1), show_y=(col == 0),
                              descriptor=(PANEL_TREATMENT_LABELS[c] if row == 0 else None))
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelsize=3.45, pad=0.7)
        if col == 0:
            ax.text(-0.27, 0.5, f"Rep. {rep}", transform=ax.transAxes, rotation=90, va="center", ha="center", fontsize=5.3)
fig.supylabel(CARBON_AXIS_LABEL, x=0.028, fontsize=7)
dominant = sorted({SPECIES[int(k)] for rr in replicate_runs.values() for k in np.unique(dominant_species_map(rr["result"]))}, key=SPECIES.index)
fig.legend(handles=species_legend_handles(dominant), loc="lower center", bbox_to_anchor=(0.52, 0.01), ncol=4, frameon=False,
           fontsize=4.45, columnspacing=0.75, handletextpad=0.3)
fig.subplots_adjust(left=0.115, right=0.995, bottom=0.20, top=0.93, hspace=0.12, wspace=0.08)
save_figure(fig, "Figure_S3_replicate_segmentations")

## Figure S4: species contributions to compositional change across state boundaries
Share of the absolute compositional change across all boundary edges of a landscape that is
carried by each species (mean +- s.e. across replicate landscapes).

In [12]:
rows = []
for (c, rep), obj in replicate_runs.items():
    labels = np.asarray(obj["result"]["labels_grid"]); grid = obj["data_array"]; contrib = np.zeros(len(SPECIES))
    for i in range(N_CARBON):
        for j in range(N_STREP):
            if i + 1 < N_CARBON and labels[i, j] != labels[i + 1, j]:
                contrib += np.abs(grid[i, j] - grid[i + 1, j])
            if j + 1 < N_STREP and labels[i, j] != labels[i, j + 1]:
                contrib += np.abs(grid[i, j] - grid[i, j + 1])
    rows += [{"community": c, "replicate": rep, "species": sp, "share": v} for sp, v in zip(SPECIES, contrib / contrib.sum())]
contributions = pd.DataFrame(rows).groupby(["community", "species"]).share.agg(["mean", "std", "count"]).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(160 * MM, 175 * MM), squeeze=False)
for ax, c in zip(axes.ravel(), COMMUNITY_TYPES):
    expected = CORE_THREE if "THREE" in c else SPECIES
    ss = contributions.loc[(contributions.community == c) & contributions.species.isin(expected)].sort_values("mean")
    y = np.arange(len(ss))
    ax.barh(y, 100 * ss["mean"], xerr=100 * ss["std"] / np.sqrt(ss["count"]), height=0.62,
            color=[SPECIES_COLORS[sp] for sp in ss.species], edgecolor="none")
    ax.set_yticks(y); ax.set_yticklabels([species_plot_label(sp) for sp in ss.species], fontsize=5.0)
    ax.text(0.5, 1.025, PANEL_TREATMENT_LABELS[c], transform=ax.transAxes, ha="center", va="bottom", fontsize=5.7)
    ax.set_xlabel("Share of compositional change (%)")
fig.subplots_adjust(top=0.90, bottom=0.08, left=0.27, right=0.98, hspace=0.36, wspace=0.48)
save_figure(fig, "Figure_S4_species_boundary_contributions")